# Music Recommendation Chatbot — Dataset Exploration

This notebook is Step 1 of the project: understand the real dataset before
writing any recommendation logic. It mirrors what `src/data_loader.py` and
`src/preprocessing.py` do, but interactively, with explanations.

Dataset: [`maharshipandya/spotify-tracks-dataset`](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset) (Hugging Face, BSD licence).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
pd.set_option('display.max_columns', 50)

from src.data_loader import load_raw_data, describe_dataset

raw_df = load_raw_data()
print(describe_dataset(raw_df))

Rows:    114,000
Columns: 20

Column names:
  track_id, artists, album_name, track_name, popularity, duration_ms, explicit, danceability, energy, key, loudness, mode, speechiness, acousticness, instrumentalness, liveness, valence, tempo, time_signature, track_genre

Columns containing missing values:
  artists: 1
  album_name: 1
  track_name: 1


## Sample rows

In [2]:
raw_df.head(5)

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## Data-quality issues found

1. **Duplicated songs across genres** — the same `track_id` appears under multiple `track_genre` values (the dataset was built by scraping ~1,000 tracks per genre playlist), so a song on two playlists is stored twice with identical audio features.

In [3]:
dupe_ids = raw_df.groupby('track_id').track_genre.nunique()
print("track_ids appearing under more than one genre:", int((dupe_ids > 1).sum()))
example_id = dupe_ids[dupe_ids > 1].index[0]
raw_df[raw_df.track_id == example_id][['track_name', 'artists', 'track_genre']]

track_ids appearing under more than one genre: 16299


,track_name,artists,track_genre
15028,Better,Pink Sweat$;Kirby,chill
103211,Better,Pink Sweat$;Kirby,soul


2. **Genre labels are an unreliable mood proxy.** The `happy` genre actually has a *below-median* mean valence, and `sad` isn't much lower. This is why the chatbot maps moods to **audio features** (danceability, energy, valence, ...) rather than to genre names.

In [4]:
mood_genres = ['happy', 'sad', 'chill', 'study', 'sleep', 'classical', 'edm']
sub = raw_df[raw_df.track_genre.isin(mood_genres)]
sub.groupby('track_genre')[['energy', 'valence', 'danceability', 'acousticness']].mean().round(3)

,energy,valence,danceability,acousticness
track_genre,,,,
chill,0.427,0.404,0.664,0.534
classical,0.190,0.381,0.382,0.920
edm,0.756,0.465,0.648,0.114
happy,0.911,0.327,0.553,0.057
sad,0.462,0.422,0.692,0.474
sleep,0.342,0.058,0.168,0.657
study,0.411,0.403,0.685,0.531


3. **Out-of-range sentinel values** — a handful of rows have `tempo == 0` or `duration_ms == 0`, which are extraction failures rather than real songs with those values.

In [5]:
print("tempo == 0 rows:", int((raw_df.tempo == 0).sum()))
print("duration_ms == 0 rows:", int((raw_df.duration_ms == 0).sum()))

tempo == 0 rows: 157
duration_ms == 0 rows: 1


4. **Loudness and tempo are on very different numeric scales** than the other seven audio features (which are already roughly 0–1). This is why scaling is required before computing any similarity — otherwise loudness/tempo would dominate the distance/cosine calculations.

In [6]:
feats = ['danceability','energy','loudness','speechiness','acousticness',
         'instrumentalness','liveness','valence','tempo']
raw_df[feats].describe().T[['min','max']]

,min,max
danceability,0.000,0.985
energy,0.000,1.000
loudness,-49.531,4.532
speechiness,0.000,0.965
acousticness,0.000,0.996
instrumentalness,0.000,1.000
liveness,0.000,1.000
valence,0.000,0.995
tempo,0.000,243.372


## Running the actual preprocessing pipeline

In [7]:
from src.preprocessing import preprocess_dataset

bundle = preprocess_dataset(raw_df)
clean_df = bundle['df']
print(f"Raw rows: {len(raw_df):,}  ->  Clean, de-duplicated songs: {len(clean_df):,}")
print("Audio features used for similarity:", bundle['audio_features'])
clean_df[['track_name', 'artists', 'genres']].head(5)

Raw rows: 114,000  ->  Clean, de-duplicated songs: 81,198
Audio features used for similarity: ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


,track_name,artists,genres
0,Comedy,Gen Hoshino,"[acoustic, j-pop, singer-songwriter, songwriter]"
1,Ghost - Acoustic,Ben Woodward,"[acoustic, chill]"
2,To Begin Again,Ingrid Michaelson;ZAYN,[acoustic]
3,Can't Help Falling In Love,Kina Grannis,[acoustic]
4,Hold On,Chord Overstreet,[acoustic]


## Quick sanity check: does content-based similarity make sense?

We use **cosine similarity on z-score standardized features** for song-to-song
similarity (see `src/preprocessing.py` and `src/recommender.py` docstrings for
why MinMax-scaled cosine doesn't discriminate well on non-negative features).
Below, we confirm a musically similar track scores much higher than a
deliberately dissimilar one.

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

ref_idx = clean_df[clean_df.track_name == 'Blinding Lights'].sort_values('popularity', ascending=False).index[0]
z = clean_df[bundle['z_columns']].to_numpy()
sims = cosine_similarity(z[ref_idx].reshape(1, -1), z)[0]

top5 = sims.argsort()[::-1]
top5 = [i for i in top5 if i != ref_idx][:5]
clean_df.loc[top5, ['track_name', 'artists', 'track_genre']].assign(similarity=sims[top5].round(3))

,track_name,artists,track_genre,similarity
10754,平凡人的自傳 - Rap Version,ONE PROMISE,cantopop,0.992
41569,Viah,Jass Manak,hip-hop,0.992
23088,Thinkin About,ShockOne;Lee Mvtthews,drum-and-bass,0.989
14852,BODY,LICK;LUNA AURA,club,0.987
23031,Fool Yourself,Chase & Status;Plan B;Rage,drum-and-bass,0.987


## Next steps

See `src/recommender.py` for the four recommendation strategies (song, mood, text, hybrid),
`src/chatbot.py` for intent detection, and `app.py` for the Streamlit UI. Run the full app with:

```bash
streamlit run app.py
```